# Calculating the mechanical properties of membranes from MD simulations


## Motivation

Lipid composition influences the elastic properties of a membrane.
The energy required to deform the membrane depends on the membrane's resistance to deformation, and the extent of that resistance is determined by its elastic properties.
Therefore, a membranes capability of generating highly curved structures is due to their inherent lipid composition.

## Helfrich-Canham-Evans Theory of Elasticity

Helfrich-Canham-Evans (HCE) theory treats the bilayer as a two-dimensional, incompressible, continuum elastic medium. When a membrane undergoes only slight deformations, this framework describes the free energy via a quadratic expansion in curvature:

$$E = \int dA \left[\frac{\kappa}{2}(2H - C_0)^2 + \bar{\kappa} K\right]$$

In summary, the free energy depends on curvature.

> **Note:** There is no tension term in the HCE model for MD simulations. This is because the membrane is continuous due to periodic boundary conditions, so surface tension is not considered.

## Spectral Analysis Approach

One can calculate the bending modulus ($\kappa$) by analyzing the excited fluctuations in the bilayer shape over the course of an MD trajectory. These fluctuations are transformed into Fourier space, decomposing the surface into waves:

$$\vec{q} = (q_x, q_y)$$

Each $\vec{q}$ corresponds to a wavelength via the inverse relationship:

$$\lambda = \frac{2\pi}{q}$$

Long wavelengths correspond to small $|\vec{q}|$, and short wavelengths to large $|\vec{q}|$.

**Physical intuition:** A stiff membrane will not have a large contribution from long-wavelength undulations (small $|\vec{q}|$), since it is much harder to bend.

**In practice to determine $\kappa$:**
1) Determine the possible $\vec{q}$ that can exist. Since in MD you are working with a box a set type of $\vec{q}$ modes are possible as it restrians the system.
2) Then determine the contribution (i.e the magnitude of each $\vec{q}$).
3) Then for each  $\vec{q}$ magnitude determine $\kappa$ from the following equation since you know all the other values. 
$$\left\langle |h_q|^2 \right\rangle = \frac{k_B T}{\kappa q^4}$$

> **Note:** We only look at small $\vec{q}$. This is because HC theory is only valid at long wavelengths. This is because as HC treats the membrane as one continuous elastic sheet and at smaller wavelengths, the membrane is not behaving as an elastic sheet therefore the theory breaks down and you cannot apply the above equation to determine the $K$.

> **Note:** As the size of the box limits what $\vec{q}$ are possible, this methodology can only be applied to larger system sizes (~1000 lipids or larger). Which makes it really hard to implement in atomistic systems that are much smaller.

## Real-Space Approach
Based on Doktorova et al. 2017 (*PCCP*), which uses real-space fluctuation methods to examine lipid tilt and splay to determine the bending rigidity of a membrane.

The continuum approach does not describe the behavior of individual lipids: Are they tilted? Stretched? Splayed? Twisted?

Rather than looking at only the overall shape of the membrane, one can examine lipid arrangements to infer the energy cost of bending.

This is a great approach when you want to look at smaller systems, but it can also be applied to the larger systems.

### Tilt and Splay Free Energy

This paper focuses on lipid **tilt** and **splay**.

The quadratic approximation for the free energy of monolayer **tilt**:

$$f_t = \frac{1}{2} k_t \langle \vec{t} \rangle^2 = \frac{1}{2} k_t \left\langle \left( \frac{\vec{n}}{\vec{n} \cdot \vec{N}} - \vec{N} \right) \right\rangle^2$$

The quadratic approximation for the free energy of monolayer **splay**:

$$f_b = \frac{1}{2} K_C \langle S \rangle^2 = \frac{1}{2} K_C \left\langle \nabla \vec{n} - \nabla \vec{N} \right\rangle^2$$

| Symbol | Definition |
|--------|-----------|
| $\vec{t}$ | Lipid tilt vector |
| $S$ | Lipid splay |
| $\vec{n}$ | Local lipid director |
| $\vec{N}$ | global normal to the lipid-water interface |
| $k_t$ | Tilt modulus |
| $K_C$ | Bending modulus |

### Tilt Modulus from Fluctuations

The tilt modulus $k_t$ is related to tilt vector fluctuations via the equipartition theorem:

$$k_t = \frac{k_B T}{A_l \langle \vec{t}^{\,2} \rangle}$$

| Symbol | Definition |
|--------|-----------|
| $A_l$ | Area per lipid |
| $\vec{t}$ | Lipid tilt vector |
| $k_B$ | Boltzmann constant (converts temperature in K to energy) |

## Something 

There are two vectors that are necessary for determing the lipid tilt and splay. 
1) $\vec{n}$
2) $\vec{N}$

### Lipid tilt 
Lipid tilt: The membrane prefers for lipids to point along the local normal, deviations from this cost energy. 

### Application of Boltzmann distribution 
If the system is assumed to be at thermal equilibrium you can represent the probability of observing a state with energy E as:
$$P(E) \propto e^{-E/(k_B T)}$$

**Physical intuition:** A low-energy configurations occur frequently while high-energy configurations occur rarely. Where the probability decreases exponentially with increasing energy.


Assuming the tilt energy is harmonic $U_s = \frac{1}{2}kx^2$, the energy of the tilt can be represented as the following:
$$ E_{tilt} = \frac{k_t \tilde t^2 A_l}{2} $$

$$ E_{Splay} = \frac{k_c \tilde S^2 A_l}{2} $$
> **Note:** You have $A_l$ the the energy equation because the $\vec{t}$ is in energy/area but you want the energy for one lipid. Therefore you multiply the area occupied by that lipid ($A_l$)

| Symbol | Definition |
|--------|-----------|
| $A_l$ | Area per lipid |
| $\vec{t}$ | Lipid tilt vector |
| $k_t$  | thermodynamic tilt modulus |

Then you can re-express the $\vec{t}$ with respect to the $\vec{n}$ and $\vec{N}$. 

$$\vec{t} = \frac{\vec{n}}{\mathbf{\vec{n}} \cdot \mathbf {\vec{N}}} -\vec{N}$$
1) You take the dot product, so that the $\vec{n}$ is normalized so it's in the membrane plane.
2) Then you can subtract $\vec{N}$ to see how much $\vec{n}$ deviates from $\vec{N}$. 

Now with the re-expressed $\vec{t}$ you can reconstruct the Boltzmann distribution.

$$P(\tilde t)=C_t\exp\!\left(-\frac{k_t \tilde t^2 A_l}{2k_B T}\right)$$


And rewrite the formula so it resembles a quadratic function. 
$$
-\frac{2k_B T}{A_l}\ln P(\tilde{t})
=
k_t
\left|
\frac{\vec{n}}
{\vec{n}\cdot\vec{N}}
-\vec{N}
\right|^2
+
C_t'
$$

To determine what $k_t$ is do the following:
1) construct a distribution of $\vec{t}$ vs $P(\vec{t})$. To determine $P(\vec{t})$.
2) Then plot $\frac{-2k_bT}{A_l}P(\vec{t})$ vs $\vec{t}$
3) Then fit a quadratic equation, where you can then pull out $k_t$.

## Adding a trick to make calculating $\vec{t}$ easier 

You will express from Cartesian coordinates to polar coordinates in tilt space. x


## Transverse Curvature Bias (TCB)

Based on Sapp et al. 2020 (*PCCP*).

General Overview: When a membrane bends, the outer leaflet is stretched and the inner leaflet is compressed. However, there exists a neutral plane between the two leaflets where neither is compressed or stretched. The placement of the lipid within the bilayer determines the extent to which it feels compressed or stretched. This outside force can bias the curvature sampled by the atoms/beads in a lipid. It is stated that the slope of z-position vs curvature bias is inversely proportional to the bending rigidity, as it costs more energy for a membrane to bend, there is not a higher deviance in the sampled curvature, so the slope is lower.

$$
\left\langle c_{\vec{q}} \right\rangle 
=
\frac{-2zk_BT}
{Area*k_c}
$$


### Describe membranes fluctuations using Fourier modes


### Geometric Density Bias

The metric factor describes the **geometric density bias** at a particular depth \(z\) within a curved membrane.

$$
m(x;z)\approx 1-zc(x)
$$

Where:

- \(m(x;z)\) is the relative apparent density of lipid positions at depth \(z\).
- \(z\) is the signed distance of an atom or bead from the neutral surface.
- \(c(x)\) is the local membrane curvature.

The product \(zc(x)\) determines how the apparent density changes with membrane curvature and transverse position.

- If \(zc(x)>0\), then \(m<1\), indicating lower apparent density.
- If \(zc(x)<0\), then \(m>1\), indicating higher apparent density.
- If \(z=0\), then \(m=1\), indicating no first-order geometric density bias.

**Importantly, the metric factor does not directly calculate curvature sampling bias.** Instead, it provides the density weighting used to determine the average curvature sampled by atoms or beads at different depths.

$$
\text{Membrane curvature}
\longrightarrow
\text{Geometric density bias}
\longrightarrow
\text{Curvature sampling bias}
$$

### Curvature Sampling Bias

To determine how geometric sampling bias affects the curvature sampled by atoms or beads at a particular depth \(z\), we integrate the local curvature weighted by the geometric density factor over the membrane surface.

$$
\int_0^{L_y}dy'\int_0^{L_x}dx'
\underbrace{\left[-\frac{\partial^2h(x,y)}{\partial x^2}\right]}_{\text{Local curvature}}
\underbrace{m(x;z)}_{\text{Geometric sampling weight}}
$$

This expression contains three components:

1. **Local curvature:** The term $-\frac{\partial^2h(x,y)}{\partial x^2}$ describes the local curvature contribution in the $x$-direction.

2. **Geometric sampling weight:** The metric factor $m(x;z)$ accounts for changes in the apparent density of lipid positions at a particular depth \(z\), depending on membrane curvature.

3. **Integration over the membrane:** The integrals over $x'$ and $y'$ sum the density-weighted curvature contributions across the membrane's projected area.

**Physical interpretation:**

Without density weighting, positive and negative curvature contributions may cancel when averaged across a nearly planar membrane.

However, because lipid positions at a particular depth may be more densely represented in some curved regions than others, the metric factor changes how strongly each region contributes to the sampled curvature.

Therefore, this integral describes how geometric sampling bias influences the total curvature sampled at depth \(z\).

**Importantly**, this integral alone does not give the final transverse curvature bias equation. The authors must also average over thermally accessible membrane fluctuations using the Boltzmann weighting factor.

This ultimately connects the depth-dependent curvature sampling bias to the membrane's bending rigidity.

### Thermal Averaging Over Membrane Fluctuations

To calculate the average curvature sampled by atoms or beads at a particular depth, we must account for the fact that the membrane is constantly fluctuating due to thermal motion.

The authors accomplish this by performing a **statistical mechanical average over all possible amplitudes of a membrane Fourier mode**.

$$
Z_q^{-1}
\int_{-\infty}^{\infty}d\operatorname{Re}(h_q)
\int_{-\infty}^{\infty}d\operatorname{Im}(h_q)
e^{-\beta E_c(h_q)}
$$

#### 1. Integration over Fourier-mode amplitudes

Membrane undulations can be represented using Fourier modes, where each mode has a complex amplitude:

$$
h_q=\operatorname{Re}(h_q)+i\operatorname{Im}(h_q)
$$

- \(\operatorname{Re}(h_q)\) is the real component of the Fourier amplitude.
- \(\operatorname{Im}(h_q)\) is the imaginary component of the Fourier amplitude.

The two integrals account for all possible values of these components, representing the possible thermal fluctuations of that Fourier mode.

#### 2. Boltzmann weighting

Not all membrane fluctuations are equally probable. Their probabilities depend on their bending energies.

The Boltzmann factor is:

$$
e^{-\beta E_c(h_q)}
$$

Where:

- \(\beta=1/(k_BT)\) is the inverse thermal energy.
- \(E_c(h_q)\) is the bending energy associated with a particular Fourier-mode amplitude.

Fluctuations with lower bending energies have greater Boltzmann weights, while fluctuations with higher bending energies have smaller weights.

Therefore, the Boltzmann factor ensures that each possible membrane fluctuation contributes according to its equilibrium probability.

#### 3. Partition function normalization

The term \(Z_q^{-1}\) is the inverse partition function for the Fourier mode.

$$
Z_q=
\int_{-\infty}^{\infty}d\operatorname{Re}(h_q)
\int_{-\infty}^{\infty}d\operatorname{Im}(h_q)
e^{-\beta E_c(h_q)}
$$

The partition function sums the Boltzmann weights over all possible Fourier-mode amplitudes.

Dividing by \(Z_q\) normalizes these weights, allowing us to calculate a statistical mechanical average.

#### 4. Connection to curvature sampling bias

Previously, we calculated the density-weighted curvature contribution across the membrane:

$$
\int_0^{L_y}dy'\int_0^{L_x}dx'
\left[-\frac{\partial^2h(x,y)}{\partial x^2}\right]m(x;z)
$$

This expression describes the spatially integrated, density-weighted curvature for a particular membrane configuration.

However, because the membrane is constantly fluctuating, we must also average this quantity over all thermally accessible configurations.

The integrals over the real and imaginary components of \(h_q\), together with the Boltzmann factor and partition function, accomplish this thermal averaging.

**Therefore, Equation 15 combines two operations:**

1. **Spatial integration:** Calculates the density-weighted curvature contribution across the membrane surface.
2. **Thermal averaging:** Accounts for the probability of different membrane fluctuations according to their bending energies.

Together, these operations determine the average curvature sampled at a particular depth within the membrane.

$$
\boxed{
\text{Curvature sampling bias}
=
\text{Thermal average of density-weighted curvature}
}
$$

**Important:** The normalized thermal integration operator alone does not calculate curvature. It must act on the curvature-dependent expression to produce the average sampled curvature.


**It is said that the curvature bias is independent of the q (wavevector magnitude).**

Curvature is related to the second spatial derivative of membrane height. $q^2$ appears because you take the second derivative.
$$ c(q) \propto q^2h(q) $$ 

To get the strength of the curvature fluctuation, regardlesss of direction we take the mean-square curvature. This allows opposing fluctuations not to cancel out and giving us a mean curvature value of zero. Taking the mean-square of the curvature fluctuations you get: 
$$ \left\langle |c(q)|^2 \right\rangle  \propto q^4 \left\langle |h(q)|^2 \right\rangle $$ 

Remember: 
$$\left\langle |h(q)|^2 \right\rangle = \frac{k_BT}
{Area*k_cq^4}
$$

Substitute, which cancels out the dependence on $q^4$
$$ \left\langle |c(q)|^2 \right\rangle  \propto q^4 \frac{k_BT}
{Area*k_cq^4} $$

Left with:
$$ \left\langle |c(q)|^2 \right\rangle  \propto \frac{k_BT}
{Area*k_c} $$
Which is the mean-square curvature fluctuations does not depend on q.




As we want how the curvature bias changes with z, we need to add a dependency on z. 


To now add the dependency on where the bead/atom is in the bilayer, which is the sampling bias.

Density of the atoms/beads change in the x-dimension dependent on the position in the z-dimension.


#### Outline of the code:
1) The bulk of the code is in the curvature_analysis.jl script 